<a href="https://colab.research.google.com/github/nmit-1NT23CS128/Social-Media-Comment-Moderation/blob/main/Social_Media_Comment_Moderation_BERT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install transformers datasets torch scikit-learn pandas numpy


In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
#from huggingface_hub import notebook_login
#notebook_login()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [3]:
import pandas as pd

data_path = "/content/drive/MyDrive/train.csv"
df = pd.read_csv(data_path)

df.head()



,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,0000997932d777bf,Explanation\nWhy the edits made under my usern...,0,0,0,0,0,0
1,000103f0d9cfb60f,D'aww! He matches this background colour I'm s...,0,0,0,0,0,0
2,000113f07ec002fd,"Hey man, I'm really not trying to edit war. It...",0,0,0,0,0,0
3,0001b41b1c6bb37e,"""\nMore\nI can't make any real suggestions on ...",0,0,0,0,0,0
4,0001d958c54c6e35,"You, sir, are my hero. Any chance you remember...",0,0,0,0,0,0


In [4]:
toxicity_columns = [
    'toxic',
    'severe_toxic',
    'obscene',
    'threat',
    'insult',
    'identity_hate'
]

df['label'] = df[toxicity_columns].max(axis=1)


In [5]:
texts = df['comment_text'].fillna("")
labels = df['label']


In [6]:
from sklearn.model_selection import train_test_split

train_texts, test_texts, train_labels, test_labels = train_test_split(
    texts,
    labels,
    test_size=0.2,
    random_state=42
)


In [7]:
print(train_labels.value_counts())


label
0    114675
1     12981
Name: count, dtype: int64


In [8]:
from transformers import DistilBertTokenizer

tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
import torch


In [ ]:
class ToxicDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer):
        self.texts = texts.reset_index(drop=True)
        self.labels = labels.reset_index(drop=True)
        self.tokenizer = tokenizer

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            str(self.texts[idx]),
            truncation=True,
            padding='max_length',
            max_length=128,
            return_tensors='pt'
        )

        item = {key: val.squeeze(0) for key, val in encoding.items()}
        item['labels'] = torch.tensor(self.labels[idx])

        return item

    def __len__(self):
        return len(self.labels)


In [ ]:
train_dataset = ToxicDataset(train_texts, train_labels, tokenizer)
test_dataset = ToxicDataset(test_texts, test_labels, tokenizer)

In [ ]:
print(len(train_dataset))
print(len(test_dataset))


In [ ]:
!pip install sympy==1.14.0 --force-reinstall
from transformers import DistilBertForSequenceClassification, Trainer, TrainingArguments

In [ ]:
model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=2
)


In [ ]:
training_args = TrainingArguments(
    output_dir='./results',

    num_train_epochs=3,

    per_device_train_batch_size=4,   # reduced
    per_device_eval_batch_size=4,    # reduced

    gradient_accumulation_steps=4,   # acts like batch size 16

    learning_rate=2e-5,

    eval_strategy="epoch",
    save_strategy="epoch",

    logging_steps=100,

    fp16=True   # ⭐ THIS IS STEP 5 (add this line)
)



In [ ]:
from transformers import Trainer


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)


In [ ]:
trainer.train()

In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

predictions = trainer.predict(test_dataset)
preds = predictions.predictions.argmax(-1)

accuracy = accuracy_score(test_labels, preds)
precision, recall, f1, _ = precision_recall_fscore_support(test_labels, preds, average='binary')

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)

In [ ]:
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,          # increase epochs
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,          # KEY CHANGE
    eval_strategy="epoch",
    save_strategy="epoch",
    weight_decay=0.01            # regularization
)


In [ ]:
from transformers import DistilBertForSequenceClassification, DistilBertTokenizer

# Re-initialize tokenizer if it's not defined (due to potential kernel restart)
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

# Re-initialize model if it's not defined (due to potential kernel restart)
model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=2
)

model.save_pretrained("toxic_model")
tokenizer.save_pretrained("toxic_model")

In [ ]:
!mkdir BERT_Toxic_Project

In [ ]:
!ls

In [ ]:
%cd BERT_Toxic_Project

In [ ]:
pip install matplotlib scikit-learn

In [ ]:
%%writefile app.py
import streamlit as st
import numpy as np
import torch
import torch.nn.functional as F
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

# Load model
model = DistilBertForSequenceClassification.from_pretrained("toxic_model")
tokenizer = DistilBertTokenizer.from_pretrained("toxic_model")
model.eval()

# Sidebar navigation
page = st.sidebar.selectbox(
    "Select Page",
    ["Toxic Comment Detector", "Model Performance"]
)

# -------------------------------
# PAGE 1: LIVE PREDICTION
# -------------------------------
if page == "Toxic Comment Detector":

    st.title("🛡 Toxic Comment Detection using BERT")

    def normalize_text(text):
        text = text.lower()
        text = text.replace("ur", "you are")
        text = text.replace("u ", "you ")
        return text

    user_input = st.text_area("Enter Comment")

    if st.button("Predict"):

        processed = normalize_text(user_input)

        inputs = tokenizer(
            processed,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=128
        )

        with torch.no_grad():
            outputs = model(**inputs)
            probs = F.softmax(outputs.logits, dim=1)
            toxic_prob = probs[0][1].item()

        if toxic_prob > 0.7:
            st.error("⚠ Toxic Comment")
        else:
            st.success("✅ Non Toxic Comment")

        st.write("Toxic Probability:", round(toxic_prob,3))
        st.progress(int(toxic_prob*100))


# -------------------------------
# PAGE 2: MODEL DASHBOARD
# -------------------------------
else:

    st.title("📊 Model Performance Dashboard")

    accuracy = 0.951
    precision = 0.763
    recall = 0.754
    f1 = 0.758

    st.metric("Accuracy", accuracy)
    st.metric("Precision", precision)
    st.metric("Recall", recall)
    st.metric("F1 Score", f1)

    # Example confusion matrix
    y_true = [0,0,0,1,1,1]
    y_pred = [0,0,1,1,1,0]

    cm = confusion_matrix(y_true, y_pred)

    fig, ax = plt.subplots()
    ax.matshow(cm)
    for (i, j), val in np.ndenumerate(cm):
        ax.text(j, i, val, ha='center', va='center')
    plt.xlabel("Predicted")
    plt.ylabel("Actual")

    st.pyplot(fig)

In [ ]:
!ls

In [ ]:
model.save_pretrained("toxic_model")
tokenizer.save_pretrained("toxic_model")

In [ ]:
!pip install pyngrok

In [ ]:
!pip install pyngrok streamlit

In [ ]:
from pyngrok import ngrok

ngrok.set_auth_token("3BDTF8NoUigVbuJIvmmnidQzfQm_4UTY1Ysq7tMr8ihd3iAC9")

In [ ]:
!streamlit run app.py &>/content/logs.txt &

In [ ]:
from pyngrok import ngrok

public_url = ngrok.connect(8501)
print(public_url)